In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/alx-essay-data/Essay_data.csv


In [18]:
from nltk.tokenize import word_tokenize
import string
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk import download
from sklearn.feature_extraction.text import CountVectorizer

In [4]:
# Load the dataset
df = pd.read_csv('/kaggle/input/alx-essay-data/Essay_data.csv', encoding='ISO-8859-1', index_col=0)
df.head()

,N/S,T/F,J/P,Essay
I/E,,,,
I,S,T,J,My first 4 months at the EDSA have been filled...
I,N,F,J,I joined the academy being at a crossroads of ...
E,N,F,J,so far my experience has been positive and i c...
I,N,F,J,I have been very fortunate to have the opportu...
I,N,T,J,Looking back to when one got to the academy an...


Tokenise the essays.

**How many tokens are in the 17th essay?**

* 128
* 94
* 440
* 337

In [8]:
# Drop rows with missing values and reset index
df = df.dropna().reset_index(drop=True)

# Function to preprocess and tokenize the text
def preprocess_and_tokenize(text):
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Convert to lower case
    text = text.lower()
    # Tokenize
    tokens = word_tokenize(text)
    return tokens

# Preprocess and tokenize the 17th essay
essay_17 = df.iloc[16]['Essay']  # Assuming the essay text is in a column named 'Essay'
tokens_17 = preprocess_and_tokenize(essay_17)
num_tokens_17 = len(tokens_17)

print(num_tokens_17)

440


Use the SnowballStemmer to stem the word experiences.

**What is the output?**

* ‘experi’
* ‘experiences’
* ‘experienc’
* ‘experience’

In [10]:
# Initialize the SnowballStemmer
stemmer = SnowballStemmer("english")

# Word to be stemmed
word = "experiences"

# Apply stemming
stemmed_word = stemmer.stem(word)

print(stemmed_word)

experi


Remove all the stop words.

**What is the 24th token in the 81st essay?**

* ‘evidently’
* ‘times’
* ‘working’
* ‘selfconfidence’

In [15]:
# Initialize stopwords
stop_words = set(stopwords.words('english'))

# Function to preprocess and remove stopwords
def preprocess_and_remove_stopwords(text):
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Convert to lower case
    text = text.lower()
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

# Preprocess and remove stopwords from the 81st essay
essay_81 = df.iloc[80]['Essay']  # Assuming the essay text is in a column named 'Essay'
tokens_81 = preprocess_and_remove_stopwords(essay_81)

# Get the 24th token
if len(tokens_81) >= 24:
    token_24 = tokens_81[23]
else:
    token_24 = None

print(token_24)

selfconfidence


**How many unique words are in these essays (after we have removed the stopwords)?**

* 3812
* 4087
* 3406
* 3494

In [16]:
# Aggregate tokens from all essays
all_tokens = []
for essay in df['Essay']:  # Assuming the essay text is in a column named 'Essay'
    tokens = preprocess_and_remove_stopwords(essay)
    all_tokens.extend(tokens)

# Find unique words
unique_words = set(all_tokens)

# Count unique words
unique_word_count = len(unique_words)
print(unique_word_count)

3454


**Create a bag of words and use it to determine how many times ‘time’ was mentioned in the 56th essay?**

* 0
* 2
* 1
* 3

In [22]:
# Create a CountVectorizer instance
vectorizer = CountVectorizer(stop_words='english')

# Fit and transform the essays
X = vectorizer.fit_transform(df['Processed_Essay'])

# Get feature names (words)
feature_names = vectorizer.get_feature_names_out()

# Find the index of the word 'time'
time_index = feature_names.tolist().index('time')

# Get the count of 'time' in the 56th essay (index 55 in 0-based index)
essay_56_vector = X[55]
time_count = essay_56_vector[0, time_index]

print(time_count)

2


**Words that appear at least twice account for what percentage of the total number of words in the essays?**

* 62%
* 90%
* 48%
* 81%

In [23]:
# Fit and transform the essays
X = vectorizer.fit_transform(df['Processed_Essay'])

# Get feature names (words) and their counts
feature_names = vectorizer.get_feature_names_out()
word_counts = X.sum(axis=0).A1
word_count_dict = dict(zip(feature_names, word_counts))

# Calculate the total number of words
total_word_count = word_counts.sum()

# Filter words that appear at least twice
words_at_least_twice = {word: count for word, count in word_count_dict.items() if count >= 2}

# Calculate the total count of words that appear at least twice
total_count_at_least_twice = sum(words_at_least_twice.values())

# Calculate the percentage
percentage = (total_count_at_least_twice / total_word_count) * 100

print(f"Percentage of total number of words accounted for by words that appear at least twice: {percentage:.2f}%")

Percentage of total number of words accounted for by words that appear at least twice: 88.22%


**What is the most commonly mentioned word by ENFJ personalities?**

* ‘academy’
* ‘data’
* ‘team’
* ‘group’

In [25]:
# Filter for ENFJ personalities
enfj_essays = df[(df['N/S'] == 'N') & (df['T/F'] == 'F') & (df['J/P'] == 'J')]['Essay']

# Create a CountVectorizer instance
vectorizer = CountVectorizer(stop_words='english')

# Fit and transform the filtered essays
X = vectorizer.fit_transform(enfj_essays)

# Get feature names (words) and their counts
feature_names = vectorizer.get_feature_names_out()
word_counts = X.sum(axis=0).A1
word_count_dict = dict(zip(feature_names, word_counts))

# Find the most commonly mentioned word
most_common_word = max(word_count_dict, key=word_count_dict.get)
most_common_word_count = word_count_dict[most_common_word]

print(f"The most commonly mentioned word by ENFJ personalities is '{most_common_word}' with {most_common_word_count} mentions.")

The most commonly mentioned word by ENFJ personalities is 'team' with 76 mentions.


Create a new column in the data frame containing the bi-grams from each essay.

**What is the 109th bi-gram in the 70th essay?**

* (‘better’, ‘certain’)
* (‘work’, ‘quite’)
* (‘may’, ‘better’)
* (‘quite’, ‘well’)

In [26]:
# Create a CountVectorizer instance for bi-grams
vectorizer = CountVectorizer(ngram_range=(2, 2), stop_words='english')

# Fit and transform the essays
X = vectorizer.fit_transform(df['Essay'])

# Get feature names (bi-grams)
feature_names = vectorizer.get_feature_names_out()

# Convert X to a DataFrame with bi-grams as columns
bi_grams_df = pd.DataFrame(X.toarray(), columns=feature_names)

# Add bi-grams to the original DataFrame
df = pd.concat([df, bi_grams_df], axis=1)

# Get the 109th bi-gram in the 70th essay
essay_index = 69  # 70th essay (index is 0-based)
bi_grams_for_essay = bi_grams_df.iloc[essay_index]
bi_grams_sorted = bi_grams_for_essay.sort_values(ascending=False)
bi_gram_list = bi_grams_sorted.index.tolist()

# Get the 109th bi-gram (index 108 because of 0-based indexing)
bi_gram_109th = bi_gram_list[108]

print(f"The 109th bi-gram in the 70th essay is: {bi_gram_109th}")

The 109th bi-gram in the 70th essay is: work teams
